## Paper Auto-Classifier 📚
### 1) Importar Librerias 

In [8]:
#Asegurate de tener activado el entorno de programación con el interprete python 3.11
# (En terminal) 
# pip install langchain_huggingface
# pip install tqdm

import pandas as pd 
import numpy as np
from LLM import Clasificador
from tqdm import tqdm

### 2) Carga BD de Bacterias 🦠🧫👨‍🔬👩‍🔬

In [9]:
#Modifica la ruta de la base de datos
#Selecona la pesataña del excel
file_path = r"E:\LLMzCor\LLMzCor.github.io\Test\Human_labeled\Clostridium_difficile.xlsx"
sheet = "Test"
usecols= [
    'PMID',
    'Title',
    'Abstract',
    #'Estado',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary',
    # 'Publication Year',
    # 'Journal/Book',
    # 'Alerta'
]
df = pd.read_excel(
    io=file_path,
    sheet_name=sheet,
    usecols=usecols, index_col=0
)

In [10]:
#df-unfilter.head()
df.shape

(50, 6)

#### Aplica filtros de Excluir (Opcional)

In [11]:
#Filtro los excluidos
# df = df[df['Estado']!="Excluir"]

df_shape = df.shape

#Calculo cuantos paper se excluyeron
resta = df .shape[0]- df.shape[0]
print("df Shape ", df_shape, " depués del filtro ", resta)

df.head()

df Shape  (50, 6)  depués del filtro  0


,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,0,1,0,Using antibodies (VHHs) AH3 and AA6 are two po...
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,0,1,0,"Administration of the PPAR-γ agonist, pioglita..."
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,0,1,0,Use Inulin or pectin as a dietary-based therap...
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,0,1,0,The paper studied a protein named PtsHN10M tha...
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,1,0,0,This study focused on analyzing Clostridioides...


#### _____________2.b) Carga de BD de Tratamientos ⚗️🧪👨‍🔬(Opcional)

In [ ]:
#Caraga la base de datos de los tratamientos (tx) para cada bacteria
treatment_path = r"E:\PaperLLM\LLMzCor.github.io\DBs\Treatment_2017.xlsx"
tx_df = pd.read_excel(io= treatment_path, index_col=0)
#tx_df.head()

#Selecciona los tratamiento de primera eleccion utilizados hasta 2017
Bacteria = "Chlamydia_trachomatis" 
treatment= tx_df.loc[Bacteria,"First-line treatment until 2017"]
print(treatment)

### 3) LLM Funciones ⚠️ Importante Ejecutar‼️

In [12]:
clasificador=Clasificador()

In [13]:
def _transformacion_binario_val(col):
    if (col=="Yes") | (col==1):
        return 1
    elif (col=="No") | (col==0):
        return 0
def _transformacion_binario_df(df):
    df[['1) Antimicrobial Resistance stain',
        '2) New treatment','3) Immunization']] = df[['1) Antimicrobial Resistance stain',
                                                     '2) New treatment',
                                                     '3) Immunization']].applymap(_transformacion_binario_val)
    return df



def ask_llm(df_,partition=0):
    df=df_.copy()
    df=_transformacion_binario_df(df)
    df["ai_label"]=np.nan
    df["ai_summary"]=np.nan
    if partition==0:
        for pmid in df.index:
            try:
                response = clasificador.clasificacion(df.loc[pmid,"Title"])
                df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df.loc[:pmid].iloc[:-1]
        return df
    else:
        print("aca")
        df_ptit=df.iloc[:partition,:]
        for pmid in df_ptit.index:
            try:
                response = clasificador.clasificacion(df_ptit.loc[pmid,"Title"])
                df_ptit.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df_ptit.loc[:pmid].iloc[:-1]
        return df_ptit

def _evaluate(row):
    values = row[["1) Antimicrobial Resistance stain","2) New treatment","3) Immunization"]].values
    ai_opcion=int(row["ai_label"])-1
    print(f"values={values}")
    print(f"ai_opcion{ai_opcion}")
    if (values.sum()==0) & (ai_opcion==3):
        return 1
    elif (values.sum()==0) & (ai_opcion<3):
        return 0
    elif (values.sum()>0) & (ai_opcion==3):
        return 0
    elif values[ai_opcion]>0:

        return 1
    else:
        return 0

def evaluacion_score(df,partition=0):
    if partition ==0:
        n=df.shape[0]
        scores = df.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    else:
        df_ptit=df.iloc[:10,:]
        n=df_ptit.shape[0]
        scores=df_ptit.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    return final_score

### _______________________Prueba clasificando un solo paper (Opcional)

In [15]:

#paper=df.loc[22304240, "Abstract"]
#paper=df.loc[22525317, "Title"]
#clasificador.clasificacion(paper)

### 4) Ask LLM to classify df 

---
Abreviatura dataframes de bacterias:
+ df --> Chlamydia trachomatis
+ df_Cd --> Clostridium difficile
+ df_Hi --> Haemopilus influenzae
+ df_Kp --> Klebsiella pneumoniae
+ df_Ng --> Neisseria gonorrhoeae
+ df_Sh --> Shigela spp
+ df_Rk --> Ricketttsia
---

###  4) Partición del df (Opcional) 🚧🚩Crhistian ver aquí 🚩🚧
Aunque df_ptit le pasa 10 o20  papers nos devuelve solo 9

### 4a) Ask_llm(df) Data Frame sin particionar

In [16]:
df_Ct = ask_llm(df)
shape_df= df_Ct.shape


print( " Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the discovery of neutralizing epitopes on Clostridioides difficile toxin A, which is a step towards immunization as it involves understanding the immune system's response to a pathogen."' has dtype incompati

Se corto el proceso del LLM por este motivo : list index out of range en el id 36026500
 Shape df_classified  (6, 8)


### 4b) Ask df_ptits Particionado

In [17]:
df_ptit=df.iloc[:10,:]
shape_df_ptit= df_ptit.shape

df_classified_1 = ask_llm(df_ptit)
shape_df= df_classified_1.shape
print("--------------------------------------------------------")

print(" The df_ptit shape is ", shape_df_ptit, ", The df_classified shape is """, shape_df )
df_classified_1[0:10][["Title","ai_label","ai_summary"]]

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the discovery of neutralizing epitopes on Clostridioides difficile toxin A, which is a step towards immunization as it involves understanding the immune system's response to a pathogen."' has dtype incompati

Se corto el proceso del LLM por este motivo : list index out of range en el id 36026500
--------------------------------------------------------
 The df_ptit shape is  (10, 6) , The df_classified shape is  (6, 8)


,Title,ai_label,ai_summary
PMID,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,3,"""The paper discusses the discovery of neutral..."
36439832,Peroxisome proliferator-activated receptor-γ a...,4,'The abstract does not discuss multiresistanc...
36439215,The impact of dietary fibers on Clostridioides...,2,'The paper discusses the impact of dietary fi...
36312948,Receptor binding protein of prophage reversibl...,4,"""The paper discusses the receptor binding pro..."
35042668,The emergence of Clostridioides difficile PCR ...,4,"""The paper discusses the emergence of a speci..."
36093337,Antibiotic resistance and genomic features of ...,1,'The abstract discusses antibiotic resistance...


In [18]:
df_ptit=df.iloc[10:20,:]
shape_df_ptit= df_ptit.shape

df_classified_2 = ask_llm(df_ptit)
shape_df= df_classified_2.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on the impact of a specific blend of human milk oligosaccharides in infant formula on gut microbiot

Shape df_ptit  (10, 6) , Shape df_classified  (10, 8)


In [20]:
df_ptit=df.iloc[20:30,:]
shape_df_ptit= df_ptit.shape

df_classified_3 = ask_llm(df_ptit)
shape_df= df_classified_3.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )
df_classified_3

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses the design, synthesis, and characterization of a new compound, TNP-2198, which has potent activity against specific types of bacteria. This falls under the category of New Treatments.'' has dtype incompatibl

Se corto el proceso del LLM por este motivo : list index out of range en el id 35283831
Shape df_ptit  (10, 6) , Shape df_classified  (4, 8)


,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
35175750,"Design, Synthesis, and Characterization of TNP...","TNP-2198, a stable conjugate of a rifamycin ph...",0,1,0,This study introduces a novel antibacterial ag...,2,"'The paper discusses the design, synthesis, a..."
35360652,Molecular Epidemiology and Antimicrobial Resis...,Clostridioides difficile is a global public he...,1,0,0,This study reveals high genetic and toxin prof...,1,'The paper discusses the antimicrobial resist...
35252669,"Development of 1,2,4-Oxadiazole Antimicrobial ...",Colonization of the gastrointestinal (GI) trac...,0,1,0,"26a an analogue from 1,2,4-oxadiazole could se...",2,"'The paper discusses the Development of 1,2,4..."
34875405,Safety and Efficacy of Prophylactic Levofloxac...,Levofloxacin has been widely used for bacterem...,0,1,0,Levofloxacin prophylaxis during the peritrans...,2,'The paper discusses the safety and efficacy ...


In [21]:
df_ptit=df.iloc[30:40,:]
shape_df_ptit= df_ptit.shape

df_classified_4 = ask_llm(df_ptit)
shape_df= df_classified_4.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses a new treatment, Myxopyronin B, which inhibits the growth of a Fidaxomicin-resistant Clostridioides difficile isolate and interferes with toxin synthesis.'' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]


Shape df_ptit  (10, 6) , Shape df_classified  (10, 8)


In [22]:
df_ptit=df.iloc[40:50,:]
shape_df_ptit= df_ptit.shape

df_classified_5 = ask_llm(df_ptit)
shape_df= df_classified_5.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the pathogenic ability of a new ribotype Clostridioides difficile from ST11 group, but it does not involve multiresistance bacteria stains report, new treatments, or immunization." ' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]


Se corto el proceso del LLM por este motivo : list index out of range en el id 34516580
Shape df_ptit  (10, 6) , Shape df_classified  (5, 8)


In [23]:
df_ptit=df.iloc[50:60,:]
shape_df_ptit= df_ptit.shape

df_classified_6 = ask_llm(df_ptit)
shape_df= df_classified_6.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

Shape df_ptit  (0, 6) , Shape df_classified  (0, 8)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


In [24]:
df_ptit=df.iloc[60:70,:]
shape_df_ptit= df_ptit.shape

df_classified_7 = ask_llm(df_ptit)
shape_df= df_classified_7.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

Shape df_ptit  (0, 6) , Shape df_classified  (0, 8)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


In [ ]:
df_ptit=df.iloc[70:80,:]
shape_df_ptit= df_ptit.shape

df_classified_8 = ask_llm(df_ptit)
shape_df= df_classified_8.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The abstract does not discuss multiresistance bacteria stains report, new treatments, or immunization. Instead, it seems to be about identifying biomarkers for trachomatous trichiasis, a condition that can lead to blindne

Se corto el proceso del LLM por este motivo : list index out of range en el id 22251247
Shape df_ptit  (10, 10) , Shape df_classified  (7, 12)


In [25]:
df_ptit=df.iloc[80:90,:]
shape_df_ptit= df_ptit.shape

df_classified_9 = ask_llm(df_ptit)
shape_df= df_classified_9.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

Shape df_ptit  (0, 6) , Shape df_classified  (0, 8)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


In [26]:
df_ptit=df.iloc[90:100,:]
shape_df_ptit= df_ptit.shape

df_classified_10 = ask_llm(df_ptit)
shape_df= df_classified_10.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

Shape df_ptit  (0, 6) , Shape df_classified  (0, 8)


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_27576\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


In [28]:
df_list= [df_classified_1, 
             df_classified_2, 
             df_classified_3, 
             df_classified_4, 
             df_classified_5, 
            #  df_classified_6, 
            #  df_classified_8, 
            #  df_classified_7, 
            #  df_classified_9, 
            #  df_classified_10
            ]
df_Cta = pd.concat(df_list, ignore_index=False)


In [30]:
print(df_Cta.shape)
df_Cta

(35, 8)


,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,0,1,0,Using antibodies (VHHs) AH3 and AA6 are two po...,3,"""The paper discusses the discovery of neutral..."
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,0,1,0,"Administration of the PPAR-γ agonist, pioglita...",4,'The abstract does not discuss multiresistanc...
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,0,1,0,Use Inulin or pectin as a dietary-based therap...,2,'The paper discusses the impact of dietary fi...
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,0,1,0,The paper studied a protein named PtsHN10M tha...,4,"""The paper discusses the receptor binding pro..."
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,1,0,0,This study focused on analyzing Clostridioides...,4,"""The paper discusses the emergence of a speci..."
36093337,Antibiotic resistance and genomic features of ...,Background: Clostridioides difficile infection...,1,0,0,While current main treatments remain effective...,1,'The abstract discusses antibiotic resistance...
35873420,Infant Formula With a Specific Blend of Five H...,Background: Human milk oligosaccharides (HMOs)...,0,0,1,Study whether infant formula supplemented with...,4,'The paper does not discuss Multiresistance b...
35844586,Corrigendum: Immunoinformatics approach toward...,"This corrects the article ""Immunoinformatics A...",0,0,0,"This corrects the article ""Immunoinformatics A...",3,'The abstract discusses an immunoinformatics ...
35217192,Clostridioides difficile from Brazilian hospit...,Clostridioides difficile (CD) is the most freq...,1,0,0,This study aimed to better understand non-toxi...,4,"""The paper's abstract focuses on the characte..."


# 7) Limpieza de Df

In [31]:
columns_excluded = ['1) Antimicrobial Resistance stain', 
                    '2) New treatment', 
                    '3) Immunization', 
                    'Abstract', 
                    'Human_summary'
                    ]

In [32]:
df_to_test= df_Cta.drop(columns= columns_excluded)

In [33]:
df_to_test

,Title,ai_label,ai_summary
PMID,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,3,"""The paper discusses the discovery of neutral..."
36439832,Peroxisome proliferator-activated receptor-γ a...,4,'The abstract does not discuss multiresistanc...
36439215,The impact of dietary fibers on Clostridioides...,2,'The paper discusses the impact of dietary fi...
36312948,Receptor binding protein of prophage reversibl...,4,"""The paper discusses the receptor binding pro..."
35042668,The emergence of Clostridioides difficile PCR ...,4,"""The paper discusses the emergence of a speci..."
36093337,Antibiotic resistance and genomic features of ...,1,'The abstract discusses antibiotic resistance...
35873420,Infant Formula With a Specific Blend of Five H...,4,'The paper does not discuss Multiresistance b...
35844586,Corrigendum: Immunoinformatics approach toward...,3,'The abstract discusses an immunoinformatics ...
35217192,Clostridioides difficile from Brazilian hospit...,4,"""The paper's abstract focuses on the characte..."


# EDA

In [34]:
print("Se analizaron ", df_to_test.shape[0], " papers.")

Se analizaron  35  papers.


#### AI Label:

1) Antimicrobial Resistance stain
2) New treatment
3) Immunization
4) None

In [35]:
df_to_test['ai_label'].value_counts()

ai_label
2    16
4    10
3     6
1     3
Name: count, dtype: int64